<a href="https://colab.research.google.com/github/silvjunu/devowel/blob/main/devowel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DeVowel

**Exploring Vowel-Reduced Vocal Timbre**

DeVowel explores whether vowel identity can be reduced from a sustained human voice while preserving speaker-specific vocal characteristics.

## Core Question

Can vowel identity be reduced while preserving the characteristic timbre of the original voice?

## Current Goal — v0.1

The first prototype will:

1. load a sustained vowel audio file,
2. inspect its waveform and spectrum,
3. estimate its spectral envelope,
4. modify the envelope to reduce vowel-specific characteristics,
5. resynthesize and listen to the processed sound.

The first version focuses on a sustained `/a/` vowel from a single speaker.

## 0. Session Setup

This section prepares the temporary Colab environment.

Because the Colab runtime can reset between sessions, every session should begin by running the setup cells from the top.

### Libraries used

- `numpy`: numerical arrays and audio signal calculations
- `scipy`: signal processing tools
- `matplotlib`: waveform and spectrum visualization
- `soundfile`: reading and writing WAV audio

At this stage, we only check that the required libraries are available. We will install additional packages only when they become necessary.

### Google Drive

Audio recordings are stored in Google Drive because the Colab runtime is temporary.

The notebook code is versioned with GitHub, while private or larger audio files are kept separately in Google Drive.

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [1]:
import numpy as np
import scipy
import matplotlib
import matplotlib.pyplot as plt
import soundfile as sf

print("Environment ready.")
print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("Matplotlib:", matplotlib.__version__)
print("SoundFile:", sf.__version__)

Environment ready.
NumPy: 2.1.3
SciPy: 1.16.3
Matplotlib: 3.10.0
SoundFile: 0.14.0


In [9]:
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/devowel")
audio_dir = project_dir / "audio"

raw_dir = audio_dir / "raw"
wav_dir = audio_dir / "wav"

wav_dir.mkdir(parents=True, exist_ok=True)

print("Raw folder:", raw_dir)
print("WAV folder:", wav_dir)
print("Raw exists:", raw_dir.exists())
print("WAV exists:", wav_dir.exists())

Raw folder: /content/drive/MyDrive/devowel/audio/raw
WAV folder: /content/drive/MyDrive/devowel/audio/wav
Raw exists: True
WAV exists: True


### Audio Format Conversion

The original recordings were captured as M4A files.

The original files are preserved in `audio/raw/`.
For analysis, they are decoded into WAV files and stored separately in `audio/wav/`.

Converting M4A to WAV does not restore information lost during compression.
WAV is used because uncompressed PCM audio is easier and more predictable to analyze.

In [8]:
!ffmpeg -version

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-l

In [11]:
import subprocess

for input_path in m4a_files:
  output_path = wav_dir / f"{input_path.stem}.wav"

  command = [
    "ffmpeg",
    "-y",
    "-i", str(input_path),
    "-ac", "1",
    "-ar", "44100",
    str(output_path),
  ]

  subprocess.run(
    command,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
    check=True,
  )

  print(f"{input_path.name} -> {output_path.name}")

a_01.m4a -> a_01.wav
a_02.m4a -> a_02.wav
e_01.m4a -> e_01.wav
e_02.m4a -> e_02.wav
i_01.m4a -> i_01.wav
i_02.m4a -> i_02.wav
o_01.m4a -> o_01.wav
o_02.m4a -> o_02.wav
u_01.m4a -> u_01.wav
u_02.m4a -> u_02.wav


In [12]:
wav_files = sorted(wav_dir.glob("*.wav"))

print("Number of WAV files:", len(wav_files))

for file in wav_files:
  print(file.name)

Number of WAV files: 10
a_01.wav
a_02.wav
e_01.wav
e_02.wav
i_01.wav
i_02.wav
o_01.wav
o_02.wav
u_01.wav
u_02.wav
